In [1]:
%run init_env.py

Added to Python path: D:\git\cs5305\genai-langchain
Environment initialization completed successfully.


## Simple Multi-Agent Design Pattern

### Prerequisite:
Define/Create all the tools required by the system

### 1. Create multiple agents, each with its own list of tools respectively...
Each subagent is then created with `create_agent(...)`:

- `subagent_1` gets access only to `square_root`
- `subagent_2` gets access only to `square`

### 2. Wrap each subagent as a tool
To let another agent use these specialists, each subagent is wrapped inside a new tool (using @tool decorator):

- `call_subagent_1(x)` sends a message to `subagent_1`
- `call_subagent_2(x)` sends a message to `subagent_2`

This means the main agent does not directly perform the math. Instead, it delegates the work to the appropriate subagent.

### 3. Create a main coordinating agent
Finally, `main_agent` is created with:

- the same LLM
- the tools `call_subagent_1` and `call_subagent_2`
- a system prompt telling it when to use them

## Creating subagents

In [2]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
llm = create_azure_llm()

agent_square_root = create_agent( model=llm, 
    tools=[square_root]
)

agent_square = create_agent( model=llm,
    tools=[square]
)

Creating Azure OpenAI LLM
  Deployment: lums-gpt-4.1-mini
  Endpoint: https://aoai-foundry-swc.openai.azure.com/
  API Version: 2025-01-01-preview
  Temperature: 0.7


In [4]:
from langchain.messages import HumanMessage

@tool
def call_agent_square_root(x: float) -> float:
    """Call agent_square_root in order to calculate the square root of a number"""
    response = agent_square_root.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_agent_square(x: float) -> float:
    """Call agent_square in order to calculate the square of a number"""
    response = agent_square.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=llm,
    tools=[call_agent_square_root, call_agent_square],
    system_prompt="You are a helpful assistant who can call subagents to " \
                   "calculate the square root or square of a number.")

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
pprint(response['messages'][-1].content)

'The square root of 456 is approximately 21.354.'


In [6]:
question = "What is the square of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
pprint(response['messages'][-1].content)

'The square of 456 is 207936.'


In [7]:
response['messages']

[HumanMessage(content='What is the square of 456?', additional_kwargs={}, response_metadata={}, id='6611340d-1b44-466a-9d8b-fecd9458b501'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 111, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSmRUWjkzOByugZcREiltalk8NjUu', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}],

In [8]:
response['messages'][1].tool_calls[0]

{'name': 'call_agent_square',
 'args': {'x': 456},
 'id': 'call_mE1zbGbjmEbD3JR94xTtYeHZ',
 'type': 'tool_call'}